In [4]:
# ── Celda 1: Cargar datos ─────────────────────────────────────────
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path("..")
df = pd.read_csv(ROOT / "data/processed/yapo_clean.csv")
df["log_precio"] = np.log(df["precio_clp"])

print(f"Dataset: {df.shape}")
print(f"Tratados (automotoras): {df['es_automotora'].sum()} ({df['es_automotora'].mean()*100:.1f}%)")
print(f"Controles (particulares): {(1-df['es_automotora']).sum()}")


Dataset: (15860, 23)
Tratados (automotoras): 4357 (27.5%)
Controles (particulares): 11503


In [10]:
df.head()

,id,titulo,precio_clp,anio,combustible,transmision,kilometraje,vendedor,region,destacado,...,comuna,marca,modelo,empresa,direccion,tipo_vendedor,descripcion,edad_auto,es_automotora,log_precio
0,32079793,BMW 320 IA AÑO 2010,5780000,2010,Bencina,Automática,122690,Juan Pablo HZ,Metropolitana,False,...,Ñuñoa,Bmw,320,Diagonal Autos,"Presidente Jose Batlle y Ordoñez 3701, Ñuñoa, ...",profesional,"BMW 320 IA 2.0 AT AÑO 2010PRECIO OCASIÓN, VEHI...",16,1,15.569914
1,32079630,2015 Peugeot 208 ALLURE E HDI 1.6,9280000,2015,Diesel,Automática,54997,Pedro Pelayo,Metropolitana,False,...,La Reina,Peugeot,208,Automoviles pelayo ltda,AVDA OSSA 315 LA REINA -,profesional,UNICO DUEÑOCOPIA DE LLAVEEXCELENTE ESTADOCIERR...,11,1,16.043372
2,32019792,Kia Frontier 2.5L 2018 Único Dueño OPORTUNIDAD,11490000,2018,Diesel,Manual,184000,MAO Autos,Metropolitana,False,...,San Miguel,Kia,Frontier,Mao Autos,"Av. José Joaquín Prieto 5650, San Miguel. -",profesional,Mao Autos¡Vendemos o recibimos tu vehículo en ...,8,1,16.256988
3,32079535,Carroceria Kia Frontier 1 Cab 2025,1790000,2025,Diesel,Otros,11111,MAO Autos,Metropolitana,False,...,San Miguel,Kia,Frontier,Mao Autos,"Av. José Joaquín Prieto 5650, San Miguel. -",profesional,"Vendemos carrocería totalmente Nueva Año 2025,...",1,1,14.397726
4,32079548,MAZDA 3 SP V 2.0 7G 6AT - 2023 | 3598,19680000,2023,Bencina,Automática,14197,Vendedor,Metropolitana,False,...,Vitacura,Mazda,3,Vendedor,NaN,profesional,"mantenciones en la marca, Con GARANTÍA 2 copi...",3,1,16.795113


 ── Celda 2: Definición formal del DAG (Pearl, 1995) ──────────────
JUSTIFICACIÓN DE CADA FLECHA:

marca → es_automotora:
 Las automotoras tienen acuerdos comerciales con ciertas marcas
(ej. concesionarios Chevrolet, Kia). Un particular no.

 marca → log_precio:
Las marcas premium (BMW, Mercedes) valen más que marcas populares.

 año → es_automotora:
#Las automotoras tienden a vender autos más nuevos (ver histograma).
#Tienen acceso a stock reciente que el particular no tiene.

 año → log_precio:
#A mayor año, mayor precio (depreciación). Efecto bien documentado.

 kilometraje → es_automotora:
Las automotoras reciben autos en parte de pago con km moderados.

kilometraje → log_precio:
A mayor km, menor precio. Efecto directo documentado.

region → es_automotora:
Las automotoras están concentradas en RM y ciudades grandes.

#region → log_precio:
El mercado en RM tiene precios distintos al resto del país.

combustible, transmision → es_automotora y → log_precio:
Las automotoras tienen más vehículos automáticos y diésel premium.
U → es_automotora y U → log_precio:
CONFOUNDER NO OBSERVABLE: calidad de mantenimiento, historial,
condición cosmética. Un auto impecable aparece en automotoras
Y en particulares, pero siempre vale más.
→ Esto INVALIDA OLS y JUSTIFICA DML con nuisance ML.
VARIABLES EXCLUIDAS DEL DAG (y por qué):
descripcion: mediador (automotoras escriben mejor PORQUE son auto.)
empresa: proxy de D, no confounder
destacado: instrumento de marketing, no confounder causal
vendedor: identificador de persona, no confounder estructural

In [9]:
from dowhy import CausalModel

# DAG SIN el nodo U (lo declaramos como latente por separado)
dag_sin_U = """
digraph {
    marca         -> es_automotora;
    marca         -> log_precio;
    modelo        -> es_automotora;
    modelo        -> log_precio;
    anio          -> es_automotora;
    anio          -> log_precio;
    kilometraje   -> es_automotora;
    kilometraje   -> log_precio;
    combustible   -> es_automotora;
    combustible   -> log_precio;
    transmision   -> es_automotora;
    transmision   -> log_precio;
    region        -> es_automotora;
    region        -> log_precio;
    es_automotora -> log_precio;
}
"""

model = CausalModel(
    data=df,
    treatment="es_automotora",
    outcome="log_precio",
    graph=dag_sin_U
)

estimand = model.identify_effect(proceed_when_unidentifiable=True)
print(estimand)

Estimand type: EstimandType.NONPARAMETRIC_ATE

### Estimand : 1
Estimand name: backdoor
Estimand expression:
       d                                                                       ↪
───────────────(E[log_precio|region,marca,combustible,kilometraje,modelo,anio, ↪
d[esₐᵤₜₒₘₒₜₒᵣₐ]                                                                ↪

↪              
↪ transmision])
↪              
Estimand assumption 1, Unconfoundedness: If U→{es_automotora} and U→log_precio then P(log_precio|es_automotora,region,marca,combustible,kilometraje,modelo,anio,transmision,U) = P(log_precio|es_automotora,region,marca,combustible,kilometraje,modelo,anio,transmision)

### Estimand : 2
Estimand name: iv
No such variable(s) found!

### Estimand : 3
Estimand name: frontdoor
No such variable(s) found!

### Estimand : 4
Estimand name: general_adjustment
Estimand expression:
       d                                                                       ↪
───────────────(E[log_precio|region,marca,comb

# ── Celda 4: Rol de U en la estrategia de estimación ─────────────
print("""
CONFOUNDER NO OBSERVABLE U — ROL EN EL MODELO
══════════════════════════════════════════════

U = "calidad no observable del auto"
    (mantenimiento, historial, condición cosmética)

¿Por qué no podemos cerrarlo con dowhy?
  dowhy solo puede verificar identificación con variables
  OBSERVADAS. U no está en el dataset → backdoor path
  D ← U → Y queda abierto en sentido formal.

¿Esto invalida el análisis?  NO, si asumimos:
  Conditional Independence Assumption (CIA):
  D ⊥ U | X   (dado que controlamos X = {marca, modelo,
               anio, km, combustible, transmision, region},
               la asignación D ya no depende de U)

  Justificación práctica:
  Dado un auto con misma marca, modelo, año y km,
  la decisión de venderlo en automotora vs particular
  NO debería depender sistemáticamente de su calidad
  no observable. Un Chevrolet Spark 2018 con 60k km
  puede aparecer en ambos grupos indistintamente.

¿Qué hace DML con esto?
  Los modelos de nuisance (Random Forest) capturan
  interacciones complejas entre los confounders X.
  Esto reduce el sesgo residual atribuible a U
  de forma más robusta que OLS.

Referencia:
  Chernozhukov et al. (2018), "Double/Debiased ML"
  The Econometrics Journal, 21(1), C1-C68.
""")


In [7]:
# ── Celda 4: Diagnóstico de positivity (common support) ───────────
# DECISIÓN: restringir a autos año >= 2005 donde hay overlap real.
# Fuente: Heckman, Ichimura & Todd (1997) — common support trimming.

print("Distribución de año por grupo:")
print(df.groupby("es_automotora")["anio"].describe().round(0))

# Observaciones con año < 2005
pre_2005 = (df["anio"] < 2005).sum()
print(f"\nAutos pre-2005: {pre_2005} ({pre_2005/len(df)*100:.1f}%)")
print("→ Estos serán excluidos en 05_features_engineering.ipynb")
print("  (positivity violation: casi no hay automotoras en ese rango)")


Distribución de año por grupo:
                 count    mean  std     min     25%     50%     75%     max
es_automotora                                                              
0              11503.0  2015.0  6.0  1990.0  2011.0  2016.0  2019.0  2026.0
1               4357.0  2021.0  4.0  1993.0  2019.0  2022.0  2023.0  2026.0

Autos pre-2005: 734 (4.6%)
→ Estos serán excluidos en 05_features_engineering.ipynb
  (positivity violation: casi no hay automotoras en ese rango)


In [11]:
# ── Celda 5: Evaluación del corte de año ─────────────────────────
# Pregunta: ¿los autos pre-2005 tienen positivity real?
# Si P(D=1 | anio < 2005) ≈ 0, hay violación de positivity.

corte = 2005
pre = df[df["anio"] < corte]
post = df[df["anio"] >= corte]

print(f"Pre-{corte}:  {len(pre)} obs | % automotoras: "
      f"{pre['es_automotora'].mean()*100:.1f}%")
print(f"Post-{corte}: {len(post)} obs | % automotoras: "
      f"{post['es_automotora'].mean()*100:.1f}%")
print()
# Regla de decisión:
# Si % automotoras pre-2005 < 5% → excluir (positivity violation)
# Si % automotoras pre-2005 >= 5% → mantener con cautela
p_pre = pre["es_automotora"].mean()
if p_pre < 0.05:
    print(f"⚠️  Solo {p_pre*100:.1f}% de automotoras pre-{corte}")
    print("   → EXCLUIR: positivity violation local")
    print("   Guardar muestra final: df_model = df[df['anio'] >= 2005]")
else:
    print(f"✅  {p_pre*100:.1f}% de automotoras pre-{corte}")
    print("   → MANTENER, pero monitorear con overlap plots")


Pre-2005:  734 obs | % automotoras: 1.9%
Post-2005: 15126 obs | % automotoras: 28.7%

⚠️  Solo 1.9% de automotoras pre-2005
   → EXCLUIR: positivity violation local
   Guardar muestra final: df_model = df[df['anio'] >= 2005]
